# GEVP subsets: the ratio of every GEVP basis of 1, 2 and 3 operators, then fits pz by pz with a chosen basis, at one (frame, q)

Same input as `gevp_3pt_ratio_fit_offforward.ipynb`: the per-q 2pt files of `delta_package_byq.py` (C3input, C2pf, C2pi; at q = 0 one file is all three) and the per-q HYP FF file of `packFF_byq.py`, for the pion or the projected proton. The 2pt files hold 6 operators, 2 smearings × 3 gammas: B = boosted [4.5, 2.25, 0.8] and U = [1.5, 1.5, 0], each with G5, G5G4 and G3G5. A GEVP basis here is any subset of these 6 operators: 6 subsets of 1 operator, 15 of 2 and 20 of 3, 41 in all.

1. **GEVP** of every subset, as section 1 of the offforward notebook: per half, $v_0$ at `t_ref` from C2pf and from C2pi; the reconstructed C2pf and C2pi, mean over t_src.
2. **3pt pieces, method 1**: the 3pt is linear in the 2pt, so the pieces of method 1 (connected $\langle C_{ab}\,O\rangle$, $\langle C_{ab}\rangle$ and $\langle O\rangle$, averaged over sources) are computed once for every element $C_{ab}$ of the 6×6 operator matrix of C3input, at one hyp and the w of `w_use`.
3. **Ratio** of every subset: its 3pt is `c_sign` $v_0(p_f)^\dagger C_{3,ab}\,v_0(p_i)$, the same as the 3pt of its reconstructed C3input (section 2 of the offforward notebook); then method 1 and the ratio of section 3 of the offforward notebook, for the direction `DIRECTION`.
3b. **Average over equivalent q** (optional, off-forward): after sections 1-3 have run at every q of `q_average_list`, the ratios and 2pts of every subset are averaged over these q, and sections 4-6 use the average.
4. **Ratio plots**: one figure per (number of operators, pz, w), one panel per subset, no fit.
5. **Choice per pz**: the GEVP basis and the tsep range of the fit, from the plots of section 4.
6. **Fits, pz by pz**: 6a the 2pt fit of the chosen basis (gap priors and $E_0$), 6b the ratio fit with the chosen basis and tsep range; bare matrix elements per pz in the layout of `fit_ratio_final_offforward.py`.

Data go to `out_dir` = `final_production/{frame}_q{qx}_{qy}_{qz}/gevp_subsets`, the plots of section 4 to `plot_root` = `compare_gevp_ratio_plots/{frame}_q{qx}_{qy}_{qz}/ratio_by_gevp_subset/{n}op/w{w}`. Each section reads what the one before saved, so after the settings cell any section can be rerun alone (6b after the settings lines of 6a).

In [38]:
#======== settings, used by every section ========
import os
import re
import json
import glob
import itertools
import h5py
import numpy as np
import matplotlib.pyplot as plt
from opt_einsum import contract

here = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta"
pack_dir = here                                             # per-q 2pt files (delta_package_byq.py): C3input, C2pf, C2pi
FF_dir = here                                               # per-q FF files (packFF_byq.py)

hadron = "pion"                                             # "pion" or "proton"
frame = "forward"                                           # "forward" (q = 0), "sym" or "asy", as in delta_package_byq.py
q = [0, 0, 0]                                               # momentum transfer in units of 2pi/L: P_f = P_i + q
q_average_list = [[2, 0, 0], [-2, 0, 0], [0, 2, 0], [0, -2, 0]]   # section 3b: equivalent q of this frame to average, each run through sections 1-3 first

#GEVP bases: every subset of op_all with n operators, n in n_op_list (6 + 15 + 20 = 41 subsets)
op_all = [(s, g) for s in range(2) for g in range(3)]      # (smear, gamma), operator a = 3 * smear + gamma: smear 0 = [4.5, 2.25, 0.8], 1 = [1.5, 1.5, 0];  gamma 0 = G5, 1 = G5G4, 2 = G3G5
smear_short = ["B", "U"]                                    # smear names in the labels: B = boosted, U = unboosted
n_op_list = [1, 2, 3,4,5,6]
t0 = 3                                                      # GEVP reference time
t_ref = 6                                                  # time of the ground-state vector v0 that reconstructs the 2pt
c_sign = {"pion": -1, "proton": +1}[hadron]                 # the pion comes out negative (the fermion-loop sign is not in the code), the projected proton positive

operator = "TXTXpTYTYm2XYXY"                                # the operator of the ratio and the fits (sections 2-6)
operator_label = {"TXTXpTYTYm2XYXY": "O3 (TXTX+TYTY-2XYXY)"}.get(operator, operator)     # in the plot titles
hyp_use = 8                                                 # the one hyp of sections 2-6
w_use = list(range(10))                                     # the w of sections 2-6
tsep_use = 19                                               # sections 2-6 keep tsep and tau = 0 .. tsep_use-1
DIRECTION = "fb"                                            # the ratio of sections 3-6: "forward", "backward" or "fb"
Ls, Lt = 32, 96

#plot style of sections 4 and 6: color, marker and fill of every tsep = tsep_style_first, tsep_style_first + 1, ... (here 0 .. 19).
#tsep 4-11: the 8 categorical colors, hollow circles; tsep 12-14: secondary, primary and muted ink (as before); tsep 0-3 and 15-19 repeat
#categorical colors with filled down-triangles and filled plus signs
tsep_style_first = 0
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
TSEP_COLOR = CAT[:4] + CAT + ["#52514e", "#0b0b0b", "#898781"] + CAT[4:] + CAT[:1]
TSEP_MARKER = ["v"] * 4 + ["o"] * 8 + ["s", "^", "D"] + ["P"] * 5
TSEP_FILLED = [True] * 4 + [False] * 9 + [True, False] + [True] * 5
STYLE = {"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
         "axes.titlesize": 7, "axes.labelsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8}


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err


def jackknife(x):
    n_cfg = x.shape[0]
    return (x.sum(axis=0) - x) / (n_cfg - 1)


def text(values):
    return [v.decode() if isinstance(v, bytes) else str(v) for v in values]


def ncfg_of(name):
    return int(name.split("_ncfg")[1].split("_")[0].split(".")[0])


def read_matrix(path):
    #one per-q 2pt file as the 6x6 operator matrix, per half: (cfg, op_snk, op_src, t_src, pz, tsep), operator a = 3 * i_smear + i_gamma (the order of op_all)
    with h5py.File(path, "r") as f:
        pt2 = {}
        for half in ["forward", "backward"]:
            if hadron == "proton" and "dirac_sink" in f.attrs["dim_pt2"]:   # open Dirac indices: positive parity, one cfg at a time (these files are several GB); _projected files are read as they are
                proj = f["parity_p"][:] if half == "forward" else -f["parity_m"][:]
                pt2[half] = np.stack([np.einsum("kl,abcdlktpT->abcdtpT", proj, f[f"pt2_{half}"][ic]) for ic in range(n_cfg)])
            else:
                pt2[half] = f[f"pt2_{half}"][:]
    matrix = {}
    for half in pt2:
        assert pt2[half].shape[1] * pt2[half].shape[3] == len(op_all), f"{path}: the smearings x gammas are not the {len(op_all)} operators of op_all"
        n_tsrc, n_pf, tsep_max = pt2[half].shape[-3:]
        matrix[half] = pt2[half].transpose(0, 1, 3, 2, 4, 5, 6, 7).reshape(n_cfg, len(op_all), len(op_all), n_tsrc, n_pf, tsep_max)
        for a, (s_snk, g_snk) in enumerate(op_all):
            for b, (s_src, g_src) in enumerate(op_all):
                assert np.array_equal(matrix[half][:, a, b], pt2[half][:, s_snk, s_src, g_snk, g_src]), f"{path}: operator pair ({a}, {b}) does not match the file"
    return matrix


#file names shared by the sections: everything of this (frame, q) goes to its own folder and has kin_tag in its name
q_tag = f"q{q[0]}_{q[1]}_{q[2]}"
kin_tag = f"{frame}_{q_tag}"
pack_file = {}
for role in ["C3input", "C2pf", "C2pi"]:                    # at q = 0 (forward) the C3input file is also the C2pf and the C2pi file
    found = glob.glob(f"{pack_dir}/2pt_*_GEVP_{kin_tag}_{'C3input' if frame == 'forward' else role}_*_{hadron}{'_projected' if hadron == 'proton' else ''}.h5")
    assert found, f"no {role} file of the {hadron} at {kin_tag} in {pack_dir}"
    pack_file[role] = max(found, key=ncfg_of)               # the file with the most cfgs
found = glob.glob(f"{FF_dir}/FF_hyp*_{q_tag}_ncfg*.h5")
assert found, f"no FF file at {q_tag} in {FF_dir}"
FF_path = max(found, key=ncfg_of)
cfg_lists = {}
for role in pack_file:
    with h5py.File(pack_file[role], "r") as f:
        cfg_lists[role] = f["cfg_list"][:]
        assert f["q"][:].tolist() == q and f.attrs["frame"] == frame, f"{pack_file[role]} is not ({frame}, q = {q})"
assert all(np.array_equal(cfg_lists[role], cfg_lists["C3input"]) for role in cfg_lists), "the C3input, C2pf and C2pi files hold different cfgs"
n_cfg = len(cfg_lists["C3input"])
n_src = int(re.search(r"_(\d+)src_", pack_file["C3input"]).group(1))
with h5py.File(pack_file["C3input"], "r") as f:
    gamma_list = text(f.attrs["gamma_list"])
    smear_list = f["smear_list"][:]                         # rows [rho_T, rho_z, mom_frac]

#the subsets, as tuples of operator indices a into op_all, in the order of itertools.combinations
subsets = {n: list(itertools.combinations(range(len(op_all)), n)) for n in n_op_list}
subset_list = [sub for n in n_op_list for sub in subsets[n]]


def subset_name(sub):
    #e.g. "BG5_UG5G4": the group name of the subset in the files
    return "_".join(f"{smear_short[op_all[a][0]]}{gamma_list[op_all[a][1]]}" for a in sub)


def subset_label(sub):
    #e.g. "{B G5, U G5G4}": the panel title
    return "{" + ", ".join(f"{smear_short[op_all[a][0]]} {gamma_list[op_all[a][1]]}" for a in sub) + "}"


gevp_tag = f"t0{t0}_tref{t_ref}"
run_tag = f"{n_src}src_ncfg{n_cfg}"


def out_dir_of(kin):
    #the data of one (frame, q) or q average
    return f"{here}/final_production/{kin}/gevp_subsets"


def plot_root_of(kin):
    #the subset plots of section 4, in {n}op/w{w}
    return f"{here}/compare_gevp_ratio_plots/{kin}/ratio_by_gevp_subset"


def gevp_path_of(kin):
    return f"{out_dir_of(kin)}/gevp_subsets_{hadron}_{kin}_{gevp_tag}_{run_tag}.h5"


def ratio_path_of(kin):
    return f"{out_dir_of(kin)}/ratio_subsets_method1_{hadron}_{kin}_{gevp_tag}_{DIRECTION}_{operator}_hyp{hyp_use}_{run_tag}.h5"


out_dir = out_dir_of(kin_tag)
plot_root = plot_root_of(kin_tag)
gevp_path = gevp_path_of(kin_tag)
ratio_path = ratio_path_of(kin_tag)
block_path = {d: f"{out_dir}/pt3_blocks_method1_{hadron}_{kin_tag}_{operator}_hyp{hyp_use}_{d}_{run_tag}.h5" for d in ("forward", "backward")}
os.makedirs(out_dir, exist_ok=True)
print(f"{hadron}, {kin_tag}: {n_cfg} cfgs")
for role in pack_file:
    print(f"  {role}: {os.path.basename(pack_file[role])}")
print(f"  FF: {os.path.basename(FF_path)}")
print(f"{len(subset_list)} GEVP subsets of " + ", ".join(f"{smear_short[s]} {gamma_list[g]}" for s, g in op_all) + f";  t0 = {t0}, t_ref = {t_ref};  output in {out_dir}")

pion, forward_q0_0_0: 183 cfgs
  C3input: 2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg183_pion.h5
  C2pf: 2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg183_pion.h5
  C2pi: 2pt_coulomb_rho4p5z2p25-rho1p5_frac0p8-0_smear_GEVP_forward_q0_0_0_C3input_fb_tsep20_3072src_ncfg183_pion.h5
  FF: FF_hyp0-8_w0-9_q0_0_0_ncfg800.h5
63 GEVP subsets of B G5, B G5G4, B G3G5, U G5, U G5G4, U G3G5;  t0 = 3, t_ref = 6;  output in /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/final_production/forward_q0_0_0/gevp_subsets


## 1. GEVP of every subset: $v_0$ and the reconstructed 2pt

As section 1 of the offforward notebook, for every subset: per half the source-averaged matrix $C(t)$ of the subset's operators (hermitian part, sign `c_sign`), of C2pf and of C2pi; GEVP $C(t)v = \lambda C(t_0) v$ on the mean of $(C_{2pf} + C_{2pi})/2$; $v_0$ = ground-state vector at `t_ref`, normalized to $v_0^\dagger C(t_0) v_0 = 1$ for that mean. One $v_0$ is used at both the sink and the source, so its phase cancels in $v_0^\dagger C_3 v_0$ and needs no convention. This is valid for forward (one matrix) and sym ($p_f$ and $p_i$ are mirror images in x, so $C(p_f) = C(p_i)$ up to noise), not for asy. A subset of one operator is that operator's 2pt divided by its value at $t_0$. Saved: $v_0$ (under both names v0_C2pf and v0_C2pi) and the reconstructed C2pf = `c_sign` $v_0^\dagger C_{2pf} v_0$ and C2pi = `c_sign` $v_0^\dagger C_{2pi} v_0$, mean over t_src.

In [39]:
#======== 1. GEVP of every subset: v0 and the reconstructed 2pt ========
def solve_gevp(C_matrix):
    #C_matrix (op, op, pz, t): eigenvalues (pz, state, t) and eigenvectors (pz, state, t, op) at t in t_list, largest eigenvalue first
    n_op = C_matrix.shape[0]
    eigen_val = np.full((n_pf, n_op, len(t_list)), np.nan)
    eigen_vec = np.full((n_pf, n_op, len(t_list), n_op), np.nan, "<c16")
    for ipf in range(n_pf):
        L = np.linalg.cholesky(C_matrix[:, :, ipf, t0])
        L_inv = np.linalg.inv(L)
        for it, t in enumerate(t_list):
            M = L_inv @ C_matrix[:, :, ipf, t] @ L_inv.conj().T
            w, u = np.linalg.eigh(M)
            v = L_inv.conj().T @ u
            order = np.argsort(w)[::-1]
            eigen_val[ipf, :, it] = w[order]
            eigen_vec[ipf, :, it] = v[:, order].T
    return eigen_val, eigen_vec


#---- the per-q files: C3input p_i -> p_f, C2pf at p_f, C2pi at p_i
with h5py.File(pack_file["C3input"], "r") as f:
    cfg_list = f["cfg_list"][:]
    t_src_list = f["t_src_list"][:]                         # (cfg, t_src)
    pf_list = f["pf_list"][:]                               # (pz, 3): p_f
    pi_list = f["pi_list"][:]                               # (pz, 3): p_i = p_f - q
momenta = {"C2pf": pf_list, "C2pi": pi_list}
for role in momenta:
    with h5py.File(pack_file[role], "r") as f:
        assert np.array_equal(f["momentum_list"][:], momenta[role]), f"{role}: momenta {f['momentum_list'][:].tolist()} are not {momenta[role].tolist()}"
        assert np.array_equal(f["t_src_list"][:], t_src_list), f"{role}: t_src differ from the C3input file"

#---- the 6x6 matrix of C2pf and C2pi, mean over t_src, per half: (cfg, op_snk, op_src, pz, tsep)
pt2_mean = {}
for role in momenta:
    same = [r for r in momenta if (r, "forward") in pt2_mean and pack_file[r] == pack_file[role]]
    if same:                                                # forward: one file for both
        for half in ["forward", "backward"]:
            pt2_mean[role, half] = pt2_mean[same[0], half]
        continue
    matrix = read_matrix(pack_file[role])                   # (cfg, op, op, t_src, pz, tsep), checked against the file
    for half in ["forward", "backward"]:
        pt2_mean[role, half] = matrix[half].mean(axis=3)
    del matrix
n_pf, tsep_max = pt2_mean["C2pf", "forward"].shape[-2:]
print("2pt matrix (cfg, op_snk, op_src, pz, tsep), mean over t_src:", pt2_mean["C2pf", "forward"].shape)

#---- hermitian part of the matrix, sign c_sign: the GEVP input
t_list = np.arange(t0 + 1, tsep_max)
it_ref = list(t_list).index(t_ref)
C = {}
for (role, half), C_half in pt2_mean.items():
    C[role, half] = c_sign * 0.5 * (C_half + np.conj(np.swapaxes(C_half, 1, 2)))
    assert (np.einsum("aapt->apt", C[role, half].mean(axis=0))[:, :, t0].real > 0).all(), f"{role} {half}: diagonal of C(t0) is not positive, check c_sign for {hadron}"

#---- every subset, per half: ONE v0 from the GEVP on the mean of (C2pf + C2pi)/2, used at the sink and at the source, so its phase cancels in v0^dag C3 v0.
#     Valid when C2pf and C2pi are the same matrix up to noise: forward (one file) and sym (p_f, p_i are x-mirror images); not asy
assert frame in ("forward", "sym"), f"frame {frame}: p_f and p_i are not mirror images, one shared v0 is not valid"
v0, pt2_rec = {}, {}
for sub in subset_list:
    idx = list(sub)                                         # the subset's rows and columns of the 6x6 matrix
    for half in ["forward", "backward"]:
        C_sub = 0.5 * (C["C2pf", half] + C["C2pi", half])[:, idx][:, :, idx].mean(axis=0)     # (op, op, pz, tsep): mean over cfgs of (C2pf + C2pi)/2
        eigen_val, eigen_vec = solve_gevp(C_sub)
        v = eigen_vec[:, 0, it_ref, :]                                             # (pz, op), v^dag C_sub(t0) v = 1
        for role in momenta:
            v0[sub, role, half] = v                                                # the same vector for C2pf and C2pi
            #reconstructed 2pt of every cfg, mean over t_src: (cfg, pz, tsep)
            pt2_rec[sub, role, half] = c_sign * np.einsum("pa,cabpT,pb->cpT", np.conj(v), pt2_mean[role, half][:, idx][:, :, idx], v)
        deviation = np.abs(0.5 * (pt2_rec[sub, "C2pf", half] + pt2_rec[sub, "C2pi", half])[:, :, t0].mean(axis=0).real - 1).max()
        assert deviation < 1e-10, f"{subset_label(sub)} {half}: v0 is not normalized to v0^dag (C2pf + C2pi)(t0)/2 v0 = 1"

#---- effective mass of the reconstructed C2pf at t = 8, forward/backward averaged: one line per subset
print(f"meff of the reconstructed C2pf at t = 8, pz = {', '.join(str(p[2]) for p in pf_list)}")
for sub in subset_list:
    c2 = 0.5 * (pt2_rec[sub, "C2pf", "forward"] + pt2_rec[sub, "C2pf", "backward"]).real          # (cfg, pz, tsep)
    meff_jk = np.log(jackknife(c2)[:, :, 8] / jackknife(c2)[:, :, 9])
    mean, err = jk(meff_jk)
    print(f"  {subset_label(sub):<26}" + " ".join(f"{m:.4f}({e * 1e4:.0f})" for m, e in zip(mean, err)))

#---- save, with the GEVP parameters
with h5py.File(gevp_path, "w") as f:
    for sub in subset_list:
        g = f.create_group(subset_name(sub))
        g.attrs["label"] = subset_label(sub)
        g.create_dataset("operators", data=np.array(sub))                         # indices a into op_all
        for role in momenta:
            for half in ["forward", "backward"]:
                g.create_dataset(f"v0_{role}_{half}", data=v0[sub, role, half])    # (pz, op)
                g.create_dataset(f"pt2_{role}_{half}", data=pt2_rec[sub, role, half])      # (cfg, pz, tsep)
    f.create_dataset("op_all", data=np.array(op_all))                             # (op, 2): (smear, gamma) of every operator a
    f.create_dataset("cfg_list", data=cfg_list)
    f.create_dataset("t_src_list", data=t_src_list)
    f.create_dataset("q", data=np.array(q))
    f.create_dataset("pf_list", data=pf_list)                                     # (pz, 3): p_f
    f.create_dataset("pi_list", data=pi_list)                                     # (pz, 3): p_i = p_f - q
    f.create_dataset("smear_list", data=smear_list)                               # rows [rho_T, rho_z, mom_frac]
    f.attrs["gamma_list"] = gamma_list
    f.attrs["hadron"] = hadron
    f.attrs["frame"] = frame
    f.attrs["dim_pt2"] = "cfg, pz, tsep: reconstructed 2pt, mean over t_src"
    f.attrs["dim_v0"] = "pz, op;  op runs over the subset's operators, in the order of its 'operators' dataset"
    f.attrs["t0"] = t0
    f.attrs["t_ref"] = t_ref
    f.attrs["c_sign"] = c_sign
    f.attrs["gevp"] = ("per subset and half: C(t) = c_sign * hermitian part of the t_src-averaged matrix of the subset's operators, of (C2pf + C2pi)/2; "
                       "GEVP C(t) v = lambda C(t0) v on its mean over cfgs (Cholesky of C(t0)); v0 = ground-state vector at t_ref, normalized to v0^dag C(t0) v0 = 1; "
                       "pt2_C2pf = c_sign v0^dag C2pf v0, pt2_C2pi = c_sign v0^dag C2pi v0 with the same v0, mean over t_src")
    f.attrs["v0_phase"] = "none needed: one v0 per subset and half, from the GEVP on (C2pf + C2pi)/2, used for both p_f and p_i, so its phase cancels in v0^dag C3 v0"
    f.attrs["source_files"] = json.dumps(pack_file)
print("saved", gevp_path)

2pt matrix (cfg, op_snk, op_src, pz, tsep), mean over t_src: (183, 6, 6, 7, 20)
meff of the reconstructed C2pf at t = 8, pz = 0, 1, 2, 3, 4, 5, 6
  {B G5}                    0.1428(4) 0.2464(4) 0.4289(11) 0.6234(31) 0.8136(87) 0.9708(201) 1.1479(470)
  {B G5G4}                  0.1410(7) 0.2502(7) 0.4392(8) 0.6344(12) 0.8200(18) 0.9897(28) 1.1363(46)
  {B G3G5}                  0.5706(63) 0.3224(7) 0.4818(7) 0.6616(8) 0.8366(13) 0.9983(22) 1.1401(34)
  {U G5}                    0.1446(2) 0.2512(3) 0.4410(9) 0.6439(30) 0.8546(106) 1.0429(427) 1.1453(1888)
  {U G5G4}                  0.1409(3) 0.2532(4) 0.4490(5) 0.6494(7) 0.8373(15) 1.0036(48) 1.1405(167)
  {U G3G5}                  0.6019(35) 0.3667(4) 0.5182(3) 0.6964(4) 0.8699(8) 1.0274(22) 1.1684(98)
  {B G5, B G5G4}            0.1413(4) 0.2444(5) 0.4235(7) 0.6141(15) 0.7991(31) 0.9632(61) 1.1201(130)
  {B G5, B G3G5}            0.1428(4) 0.2462(4) 0.4269(8) 0.6173(15) 0.8020(32) 0.9640(61) 1.1112(123)
  {B G5, U G5}              0.

## 2. 3pt pieces of every operator pair, method 1

The 3pt of a subset is `c_sign` $\sum_{ab} v_{0,a}(p_f)^* \,C_{3,ab}\, v_{0,b}(p_i)$, where $C_{3,ab}$ is the 3pt built from the element $C_{ab}$ of the C3input matrix, because $v_0$ is the same for every cfg and t_src. So the pieces of method 1 are computed here once for all 36 elements $C_{ab}$, per cfg (the jackknife comes in section 3, after $v_0$): the connected $\langle C_{ab}(t_{\rm src})\,O(t_{\rm src} \pm \tau)\rangle$, $\langle C_{ab}\rangle$ and $\langle O\rangle$, each averaged over the t_src. The operator is read at `hyp_use` and the w of `w_use`, with the centering phase $e^{-i\pi q_z w/L_s}$ of the offforward notebook; tsep and $\tau$ run over $0 \ldots$ `tsep_use`$-1$.

In [29]:
#======== 2. 3pt pieces of every operator pair, method 1 ========
#---- the C3input matrix per t_src: (cfg, op_snk, op_src, t_src, pz, tsep), sink at p_f, source at p_i
with h5py.File(pack_file["C3input"], "r") as f:
    cfg_list = f["cfg_list"][:]
    t_src_list = f["t_src_list"][:]
    pf_list = f["pf_list"][:]
    pi_list = f["pi_list"][:]
pt2_C3 = read_matrix(pack_file["C3input"])
n_cfg, n_all, _, n_tsrc, n_pf, tsep_max = pt2_C3["forward"].shape
assert tsep_use <= tsep_max, f"tsep_use = {tsep_use} is beyond the {tsep_max} tsep of the 2pt"

#---- the three FF components at hyp_use and the w of w_use, on all Lt slices, on the cfgs of the 2pt: (cfg, w, t); FF munu order [XY, YZ, XZ, XT, YT, ZT]
with h5py.File(FF_path, "r") as f:
    assert f["q"][:].tolist() == q, f"the FF file holds q = {f['q'][:].tolist()}, not q = {q}"
    FF_cfg_list = f["cfg_list"][:].tolist()
    ihyp = list(f["hyp_list"][:]).index(hyp_use)
    iw = [list(f["w_list"][:]).index(w) for w in w_use]
    FF_smearing = str(f.attrs["smearing"])
    idx = [FF_cfg_list.index(int(c)) for c in cfg_list]                    # increasing, as h5py needs
    O_full = {name: f["FF"][idx, mu, mu, ihyp][:, iw] for name, mu in [("TXTX", 3), ("TYTY", 4), ("XYXY", 0)]}     # (cfg, w, t)
#centering: the FF line runs from x to x - w z with the Fourier phase at x; centered at x - w z/2 the operator has exp(-i pi q_z w / Ls) more (1 for q_z = 0)
centering = np.exp(-1j * np.pi * q[2] * np.array(w_use) / Ls)
for name in O_full:
    O_full[name] = O_full[name] * centering[None, :, None]
O_full["TXTXpTYTY"] = O_full["TXTX"] + O_full["TYTY"]
O_full["TXTXpTYTYm2XYXY"] = O_full["TXTXpTYTY"] - 2 * O_full["XYXY"]
O_full["TXTXpTYTYp2XYXY"] = O_full["TXTXpTYTY"] + 2 * O_full["XYXY"]
O = O_full[operator]

tau = np.arange(tsep_use)
for d, sign in [("forward", +1), ("backward", -1)]:
    #operator at the 2pt source times: O(t_src + tau) forward, O(t_src - tau) backward, (cfg, w, t_src, tau)
    t_index = (t_src_list[:, :, None] + sign * tau[None, None, :]) % Lt    # (cfg, t_src, tau)
    op = np.stack([O[ic][:, t_index[ic]] for ic in range(n_cfg)])
    pt2_src = pt2_C3[d][..., :tsep_use]                                     # (cfg, op_snk, op_src, t_src, pf, tsep)

    #connected piece and the two averages of method 1, mean over t_src, per cfg
    connected = contract(pt2_src, ["cfg", "a", "b", "tsrc", "pf", "tsep"],
                         op,      ["cfg", "w", "tsrc", "tau"],
                                  ["cfg", "a", "b", "w", "tsep", "tau", "pf"]) / n_tsrc
    pt2_avg = pt2_src.mean(axis=3)                                          # (cfg, op_snk, op_src, pz, tsep)
    op_avg = op.mean(axis=2)                                                # (cfg, w, tau)

    with h5py.File(block_path[d], "w") as f:
        f.create_dataset("connected", data=connected)                       # (cfg, op_snk, op_src, w, tsep, tau, pz)
        f.create_dataset("pt2_avg", data=pt2_avg)                           # (cfg, op_snk, op_src, pz, tsep)
        f.create_dataset("op_avg", data=op_avg)                             # (cfg, w, tau)
        f.create_dataset("op_all", data=np.array(op_all))                   # (op, 2): (smear, gamma) of every operator a
        f.create_dataset("w_list", data=np.array(w_use, dtype=np.int64))
        f.create_dataset("q", data=np.array(q))
        f.create_dataset("pf_list", data=pf_list)                           # (pz, 3): p_f
        f.create_dataset("pi_list", data=pi_list)                           # (pz, 3): p_i = p_f - q
        f.create_dataset("cfg_list", data=cfg_list)
        f.create_dataset("t_src_list", data=t_src_list)
        f.attrs["forward_or_backward"] = d
        f.attrs["frame"] = frame
        f.attrs["operator"] = operator
        f.attrs["hyp"] = hyp_use
        f.attrs["dim_connected"] = "cfg, op_snk, op_src, w, tsep, tau, pz"
        f.attrs["pieces"] = (f"per cfg, mean over the {n_tsrc} t_src: connected = <C_ab(t_src, tsep) O(t_src {'+' if sign > 0 else '-'} tau)>, pt2_avg = <C_ab>, op_avg = <O>, "
                             "C_ab = element (a, b) of the C3input operator matrix as in the file (no c_sign); "
                             "method 1 of a subset: jackknife of c_sign v0^dag connected v0 minus jackknife of c_sign v0^dag pt2_avg v0 times jackknife of op_avg")
        f.attrs["smearing"] = FF_smearing
        f.attrs["FF_source"] = FF_path
        f.attrs["pt2_source"] = pack_file["C3input"]
        f.attrs["centering"] = "exp(-i pi q_z w / Ls) applied to the FF: operator centered at x - w z/2"
    print("saved", block_path[d])
    del connected, op, pt2_src
del pt2_C3

saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/final_production/sym_q0_-2_0/gevp_subsets/pt3_blocks_method1_pion_sym_q0_-2_0_TXTXpTYTYm2XYXY_hyp8_forward_3072src_ncfg183.h5
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/final_production/sym_q0_-2_0/gevp_subsets/pt3_blocks_method1_pion_sym_q0_-2_0_TXTXpTYTYm2XYXY_hyp8_backward_3072src_ncfg183.h5


## 3. Ratio of every subset

Per subset and half, the 3pt pieces of section 2 are contracted with that half's $v_0(p_f)$ at the sink and $v_0(p_i)$ at the source (sign `c_sign`); then method 1 as in section 2 of the offforward notebook (jackknife of the connected piece minus jackknife $\langle C_2\rangle$ times jackknife $\langle O\rangle$) and the ratio of its section 3,

$$R = \frac{\mathrm{Re}\,C_3(t_{\rm sep},\tau)}{C_2(p_f,t_{\rm sep})}\sqrt{\frac{C_2(p_i,t_{\rm sep}-\tau)\,C_2(p_f,\tau)\,C_2(p_f,t_{\rm sep})}{C_2(p_f,t_{\rm sep}-\tau)\,C_2(p_i,\tau)\,C_2(p_i,t_{\rm sep})}}$$

with $C_2$ the real part of the subset's reconstructed C2pf and C2pi of section 1 (at q = 0 the root is 1), on every jackknife replicate, for `DIRECTION` (fb = ½(forward + backward) of the 3pt and 2pt jackknife lists); τ > tsep is nan. The same ratio with Im $C_3$ is saved as a check (also per jackknife replicate, for section 3b), and its largest |value| / error is printed per subset. All subsets go to one file, one group per subset.

In [30]:
#======== 3. ratio of every subset ========
#---- the 3pt pieces of section 2 per direction, and the GEVP of section 1
blocks = {}
for d in ("forward", "backward"):
    with h5py.File(block_path[d], "r") as f:
        blocks[d] = {name: f[name][:] for name in ("connected", "pt2_avg", "op_avg")}
        cfg_list = f["cfg_list"][:]
        pf_list = f["pf_list"][:]
        pi_list = f["pi_list"][:]
        assert f["w_list"][:].tolist() == w_use, f"{block_path[d]} holds w = {f['w_list'][:].tolist()}, not w_use = {w_use}"
        vacuum_note = str(f.attrs["pieces"])
v0, c2f, c2i = {}, {}, {}
with h5py.File(gevp_path, "r") as f:
    assert np.array_equal(f["cfg_list"][:], cfg_list), "the 3pt pieces and the GEVP have different cfgs"
    for sub in subset_list:
        g = f[subset_name(sub)]
        for d in ("forward", "backward"):
            v0[sub, "C2pf", d], v0[sub, "C2pi", d] = g[f"v0_C2pf_{d}"][:], g[f"v0_C2pi_{d}"][:]          # (pz, op)
            c2f[sub, d] = jackknife(g[f"pt2_C2pf_{d}"][:, :, :tsep_use].real)                         # (jk, pz, tsep)
            c2i[sub, d] = jackknife(g[f"pt2_C2pi_{d}"][:, :, :tsep_use].real)
    gevp_note = str(f.attrs["gevp"])
op_jk = {d: jackknife(blocks[d]["op_avg"]) for d in blocks}                                            # (jk, w, tau)

#---- R as in section 3 of the offforward notebook; tau counts from the source: p_i lies between source and operator, p_f between operator and sink
tsep_grid, tau_grid = np.meshgrid(np.arange(tsep_use), np.arange(tsep_use), indexing="ij")    # (tsep, tau)
inside = tau_grid <= tsep_grid
gap = np.where(inside, tsep_grid - tau_grid, 0)                            # tsep - tau (0 outside, masked below)
with h5py.File(ratio_path, "w") as fout:
    for sub in subset_list:
        idx = list(sub)
        pt3, f2, i2 = {}, {}, {}
        for d in ("forward", "backward"):
            v_snk, v_src = np.conj(v0[sub, "C2pf", d]), v0[sub, "C2pi", d]
            connected = c_sign * np.einsum("pa,cabwTtp,pb->cwTtp", v_snk, blocks[d]["connected"][:, idx][:, :, idx], v_src)   # (cfg, w, tsep, tau, pz)
            pt2_avg = c_sign * np.einsum("pa,cabpT,pb->cpT", v_snk, blocks[d]["pt2_avg"][:, idx][:, :, idx], v_src)          # (cfg, pz, tsep)
            vac_jk = contract(jackknife(pt2_avg), ["jk", "pf", "tsep"],
                              op_jk[d],           ["jk", "w", "tau"],
                                                  ["jk", "w", "tsep", "tau", "pf"])
            pt3[d] = jackknife(connected) - vac_jk                          # (jk, w, tsep, tau, pz)
            f2[d], i2[d] = c2f[sub, d], c2i[sub, d]
        for x in (pt3, f2, i2):
            x["fb"] = 0.5 * (x["forward"] + x["backward"])                  # replicate by replicate: same cfg_list, same left-out cfg
        F, I = f2[DIRECTION], i2[DIRECTION]
        with np.errstate(invalid="ignore"):                                 # a negative 2pt at large t gives nan
            root = np.sqrt(I[:, :, gap] * F[:, :, tau_grid] * F[:, :, tsep_grid] / (F[:, :, gap] * I[:, :, tau_grid] * I[:, :, tsep_grid]))   # (jk, pz, tsep, tau)
        factor = np.where(inside, root / F[:, :, tsep_grid], np.nan).transpose(0, 2, 3, 1)       # (jk, tsep, tau, pz)
        ratio_jk = pt3[DIRECTION].real * factor[:, None]                    # (jk, w, tsep, tau, pz)
        ratio_mean, ratio_err = jk(ratio_jk)
        ratio_imag_jk = pt3[DIRECTION].imag * factor[:, None]               # the same ratio with Im C3: a check (section 3b averages it over q)
        imag_mean, imag_err = jk(ratio_imag_jk)
        with np.errstate(invalid="ignore", divide="ignore"):
            pull = max(np.nanmax(np.abs(imag_mean[:, tsep, tau] / imag_err[:, tsep, tau])) for tsep in range(4, tsep_use) for tau in range(2, tsep - 1))
        print(f"{subset_label(sub):<26} largest |Im R| / error at tsep 4-{tsep_use - 1}, 2 <= tau <= tsep - 2, all w, all pz: {pull:.1f}")

        g = fout.create_group(subset_name(sub))
        g.attrs["label"] = subset_label(sub)
        g.create_dataset("operators", data=np.array(sub))                  # indices a into op_all
        g.create_dataset("ratio_jk", data=ratio_jk)                         # (jk, w, tsep, tau, pz)   nan for tau > tsep
        g.create_dataset("ratio_mean", data=ratio_mean)
        g.create_dataset("ratio_err", data=ratio_err)
        g.create_dataset("ratio_imag_jk", data=ratio_imag_jk)
        g.create_dataset("ratio_imag_mean", data=imag_mean)
        g.create_dataset("ratio_imag_err", data=imag_err)
    fout.create_dataset("op_all", data=np.array(op_all))
    fout.create_dataset("w_list", data=np.array(w_use, dtype=np.int64))
    fout.create_dataset("q", data=np.array(q))
    fout.create_dataset("pf_list", data=pf_list)                           # (pz, 3): p_f
    fout.create_dataset("pi_list", data=pi_list)                           # (pz, 3): p_i = p_f - q
    fout.create_dataset("cfg_list", data=cfg_list)
    fout.attrs["dim_ratio_jk"] = "jk, w, tsep, tau, pz"
    fout.attrs["ratio_definition"] = ("R = Re C3(tsep,tau) / C2(p_f,tsep) * sqrt[C2(p_i,tsep-tau) C2(p_f,tau) C2(p_f,tsep) / (C2(p_f,tsep-tau) C2(p_i,tau) C2(p_i,tsep))], "
                                      "C2 = Re of the subset's reconstructed C2pf and C2pi, mean over t_src; nan for tau > tsep")
    fout.attrs["operator"] = operator
    fout.attrs["hyp"] = hyp_use
    fout.attrs["hadron"] = hadron
    fout.attrs["frame"] = frame
    fout.attrs["pt2"] = f"GEVP {gevp_tag}: " + gevp_note
    fout.attrs["vacuum_method"] = 1
    fout.attrs["vacuum_subtraction"] = vacuum_note
    fout.attrs["direction"] = DIRECTION + ("" if DIRECTION != "fb" else ": 0.5 * (forward + backward) of the 3pt and 2pt jackknife lists, then the ratio")
    fout.attrs["pt3_source"] = json.dumps(block_path)
    fout.attrs["pt2_source"] = gevp_path
print("saved", ratio_path)
del blocks

{B G5}                     largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.5
{B G5G4}                   largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.8
{B G3G5}                   largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.1
{U G5}                     largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.2
{U G5G4}                   largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 4.1
{U G3G5}                   largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 2.8
{B G5, B G5G4}             largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.7
{B G5, B G3G5}             largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.9
{B G5, U G5}               largest |Im R| / error at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.6
{B G5, U G5G4}             largest |Im R| / error at ts

## 3b. Average over equivalent q (optional)

As section 3b of the offforward notebook: for the q of `q_average_list` (same frame, the same $|p_f|$ and $|p_i|$ at every pz), each run through sections 1-3 first (set `q` in the settings and run sections 1-3 once per q). This cell averages over these q, replicate by replicate, every subset's ratio and Im ratio and its reconstructed C2pf and C2pi, and saves them in the layouts of sections 1 and 3 under `final_production/{frame}_avg_{q tags}/gevp_subsets`. Afterwards `q`, `kin_tag`, `out_dir`, `plot_root`, `gevp_path` and `ratio_path` are those of the average, so sections 4-6 plot and fit it; run the settings cell again to go back to one q. The labels of the average (`q`, `pf_list`, `pi_list`) are those of the first q of the list.

In [31]:
#======== 3b. average over equivalent q (optional) ========
assert q_average_list, "q_average_list is empty: nothing to average"
kin_in = [f"{frame}_q{qq[0]}_{qq[1]}_{qq[2]}" for qq in q_average_list]
missing = [name for kin in kin_in for name in (gevp_path_of(kin), ratio_path_of(kin)) if not os.path.exists(name)]
assert not missing, "run sections 1-3 at every q of q_average_list first; missing:\n" + "\n".join(missing)
n_q_avg = len(q_average_list)

#---- check the inputs: the same cfgs and the same |p_f|, |p_i| at every pz
for kin in kin_in:
    with h5py.File(gevp_path_of(kin), "r") as f:
        if kin == kin_in[0]:
            cfg_list, t_src_list, pf_list, pi_list = f["cfg_list"][:], f["t_src_list"][:], f["pf_list"][:], f["pi_list"][:]
            gevp_note = str(f.attrs["gevp"])
        assert np.array_equal(f["cfg_list"][:], cfg_list) and np.array_equal(f["t_src_list"][:], t_src_list), f"{kin}: other cfgs or t_src than {kin_in[0]}"
        assert (np.sum(f["pf_list"][:] ** 2, axis=1) == np.sum(pf_list ** 2, axis=1)).all() and (np.sum(f["pi_list"][:] ** 2, axis=1) == np.sum(pi_list ** 2, axis=1)).all(), \
            f"{kin}: |p_f| or |p_i| differ from {kin_in[0]}, so it is not an equivalent q"
    with h5py.File(ratio_path_of(kin), "r") as f:
        assert np.array_equal(f["cfg_list"][:], cfg_list), f"{kin}: the ratio has other cfgs than {kin_in[0]}"
        assert f["w_list"][:].tolist() == w_use, f"{ratio_path_of(kin)} holds w = {f['w_list'][:].tolist()}, not w_use = {w_use}"
        ratio_note = str(f.attrs["ratio_definition"])

#---- from here on the names are those of the average
q = [int(v) for v in q_average_list[0]]
kin_tag = f"{frame}_avg_" + "-".join(f"q{qq[0]}_{qq[1]}_{qq[2]}" for qq in q_average_list)
out_dir = out_dir_of(kin_tag)
plot_root = plot_root_of(kin_tag)
gevp_path = gevp_path_of(kin_tag)
ratio_path = ratio_path_of(kin_tag)
os.makedirs(out_dir, exist_ok=True)

#---- every subset: mean over q of the reconstructed 2pt (the jackknife is linear: this is the mean of the replicates) and of the ratio replicates
with h5py.File(gevp_path, "w") as fgevp, h5py.File(ratio_path, "w") as fratio:
    for sub in subset_list:
        pt2_avg, ratio_jk, ratio_imag_jk = {}, 0, 0
        for kin in kin_in:
            with h5py.File(gevp_path_of(kin), "r") as f:
                for role in ("C2pf", "C2pi"):
                    for d in ("forward", "backward"):
                        pt2_avg[role, d] = pt2_avg.get((role, d), 0) + f[subset_name(sub)][f"pt2_{role}_{d}"][:] / n_q_avg      # (cfg, pz, tsep)
            with h5py.File(ratio_path_of(kin), "r") as f:
                ratio_jk = ratio_jk + f[subset_name(sub)]["ratio_jk"][:] / n_q_avg                 # (jk, w, tsep, tau, pz)
                ratio_imag_jk = ratio_imag_jk + f[subset_name(sub)]["ratio_imag_jk"][:] / n_q_avg
        g = fgevp.create_group(subset_name(sub))
        g.attrs["label"] = subset_label(sub)
        g.create_dataset("operators", data=np.array(sub))
        for (role, d), x in pt2_avg.items():
            g.create_dataset(f"pt2_{role}_{d}", data=x)
        ratio_mean, ratio_err = jk(ratio_jk)
        imag_mean, imag_err = jk(ratio_imag_jk)
        with np.errstate(invalid="ignore", divide="ignore"):
            pull = max(np.nanmax(np.abs(imag_mean[:, tsep, tau] / imag_err[:, tsep, tau])) for tsep in range(4, tsep_use) for tau in range(2, tsep - 1))
        print(f"{subset_label(sub):<26} largest |Im R| / error of the average at tsep 4-{tsep_use - 1}, 2 <= tau <= tsep - 2, all w, all pz: {pull:.1f}")
        g = fratio.create_group(subset_name(sub))
        g.attrs["label"] = subset_label(sub)
        g.create_dataset("operators", data=np.array(sub))
        for name, x in [("ratio_jk", ratio_jk), ("ratio_mean", ratio_mean), ("ratio_err", ratio_err),
                        ("ratio_imag_jk", ratio_imag_jk), ("ratio_imag_mean", imag_mean), ("ratio_imag_err", imag_err)]:
            g.create_dataset(name, data=x)
    for f in (fgevp, fratio):
        f.create_dataset("op_all", data=np.array(op_all))
        f.create_dataset("cfg_list", data=cfg_list)
        f.create_dataset("q", data=np.array(q))
        f.create_dataset("q_average_list", data=np.array(q_average_list))
        f.create_dataset("pf_list", data=pf_list)                           # (pz, 3): labels of the first q
        f.create_dataset("pi_list", data=pi_list)
        f.attrs["hadron"] = hadron
        f.attrs["frame"] = frame
        f.attrs["source_files"] = json.dumps([gevp_path_of(kin) if f is fgevp else ratio_path_of(kin) for kin in kin_in])
    fgevp.create_dataset("t_src_list", data=t_src_list)
    fgevp.create_dataset("smear_list", data=smear_list)
    fgevp.attrs["gamma_list"] = gamma_list
    fgevp.attrs["dim_pt2"] = "cfg, pz, tsep: reconstructed 2pt, mean over t_src"
    fgevp.attrs["average"] = ("mean over q_average_list of the reconstructed C2pf and C2pi of section 1 (each q with its own GEVP vectors); "
                              "q, pf_list and pi_list are the labels of the first q (every q of the list has the same |p_f| and |p_i|)")
    fgevp.attrs["gevp"] = gevp_note
    fratio.create_dataset("w_list", data=np.array(w_use, dtype=np.int64))
    fratio.attrs["dim_ratio_jk"] = "jk, w, tsep, tau, pz"
    fratio.attrs["ratio_definition"] = "mean over q_average_list of the ratios of section 3, replicate by replicate; per q: " + ratio_note
    fratio.attrs["operator"] = operator
    fratio.attrs["hyp"] = hyp_use
    fratio.attrs["direction"] = DIRECTION
print("saved", gevp_path)
print("saved", ratio_path)

{B G5}                     largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.4
{B G5G4}                   largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 4.0
{B G3G5}                   largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.6
{U G5}                     largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.6
{U G5G4}                   largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 4.1
{U G3G5}                   largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 2.9
{B G5, B G5G4}             largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 4.3
{B G5, B G3G5}             largest |Im R| / error of the average at tsep 4-18, 2 <= tau <= tsep - 2, all w, all pz: 3.9
{B G5, U G5}               largest |Im R

## 4. Ratio plots of every subset

One figure per (number of operators, pz, w), saved as `plot_root/{n}op/w{w}/ratio_gevp_{n}op_{hadron}_{kin_tag}_pz{pz}_w{w}.png`; one panel per subset, in the order of `itertools.combinations`, and the legend in the first free slot. Each panel shows the ratio of section 3 (or 3b) against τ − tsep/2 for the tsep of `PLOT_TSEP` at this pz; no fit. Every tsep has its own color and marker in all figures. The y range is shared by the figures of 1, 2 and 3 operators at the same (pz, w): it covers the point means whose error is below `Y_ERR_MAX`, so a very noisy point can run off the panel. These plots are for choosing the GEVP basis and the tsep range of each pz in section 5.

In [33]:
#======== 4. ratio plots: one figure per (number of operators, pz, w), one panel per subset ========
PLOT_TSEP = {0: (4, 18), 1: (4, 18), 2: (4, 15), 3: (4, 14), 4: (4, 14), 5: (4, 12), 6: (4, 12)}   # (first, last) tsep plotted at each pz = p_f[2]
w_plot_list = [0, 2, 4, 6, 8]
PANEL_SIZE = (3.75, 3.45)                                   # inches per panel
DPI = 110
Y_ERR_MAX = 0.25                                            # the y range covers the point means whose error is below this

#---- the ratio of section 3 (or 3b) of every subset
ratio_mean, ratio_err = {}, {}
with h5py.File(ratio_path, "r") as f:
    pf_list, pi_list = f["pf_list"][:], f["pi_list"][:]
    n_cfg_plot = len(f["cfg_list"][:])
    w_list = f["w_list"][:].tolist()
    for sub in subset_list:
        ratio_mean[sub], ratio_err[sub] = f[subset_name(sub)]["ratio_mean"][:], f[subset_name(sub)]["ratio_err"][:]      # (w, tsep, tau, pz)
assert all(last < tsep_use for first, last in PLOT_TSEP.values()), f"PLOT_TSEP goes beyond tsep_use = {tsep_use}"
assert all(w in w_list for w in w_plot_list), f"w_plot_list {w_plot_list} is not inside the w of the ratio, {w_list}"
assert all(0 <= first - tsep_style_first and last - tsep_style_first < len(TSEP_COLOR) for first, last in PLOT_TSEP.values()), \
    f"PLOT_TSEP needs tsep {tsep_style_first} .. {tsep_style_first + len(TSEP_COLOR) - 1}, the tsep with a style"
x_shift = 0.04 * (np.arange(len(TSEP_COLOR)) + tsep_style_first - 7)     # small shift per tsep, so that the error bars of different tsep do not overlap
legend_text = ", ".join(f"{smear_short[i]} = {'boosted ' if row[2] else ''}[{row[0]:g},{row[1]:g},{row[2]:g}]" for i, row in enumerate(smear_list))
if "_avg_" in kin_tag:                                     # after section 3b
    legend_text += f"\nmean over q = " + ", ".join(f"({qq[0]},{qq[1]},{qq[2]})" for qq in q_average_list)

#---- y range of every (pz, w), shared by the figures of all numbers of operators
ylim = {}
for ipf, pf in enumerate(pf_list):
    first, last = PLOT_TSEP[pf[2]]
    for w in w_plot_list:
        iw = w_list.index(w)
        values = [0.0]
        for sub in subset_list:
            for tsep in range(first, last + 1):
                mean, err = ratio_mean[sub][iw, tsep, :tsep + 1, ipf], ratio_err[sub][iw, tsep, :tsep + 1, ipf]
                values += list(mean[err < Y_ERR_MAX])
        lo, hi = min(values), max(values)
        ylim[ipf, w] = (lo - 0.05 * (hi - lo), hi + 0.05 * (hi - lo))

#---- the figures
with plt.rc_context(STYLE):
    for n in n_op_list:
        n_sub = len(subsets[n])
        n_col = 4 if n_sub + 1 <= 16 else 6
        n_row = -(-(n_sub + 1) // n_col)                    # room for the legend after the last subset
        for w in w_plot_list:
            iw = w_list.index(w)
            os.makedirs(f"{plot_root}/{n}op/w{w}", exist_ok=True)
            for ipf, (pf, pi) in enumerate(zip(pf_list, pi_list)):
                first, last = PLOT_TSEP[pf[2]]
                fig, axes = plt.subplots(n_row, n_col, figsize=(PANEL_SIZE[0] * n_col, PANEL_SIZE[1] * n_row), sharey=True, squeeze=False)
                axes = axes.ravel()
                for i, sub in enumerate(subsets[n]):
                    ax = axes[i]
                    ax.axhline(0, color="#52514e", lw=0.8, zorder=1)
                    for tsep in range(first, last + 1):
                        it = tsep - tsep_style_first
                        tau = np.arange(tsep + 1)
                        ax.errorbar(tau - tsep / 2 + x_shift[it], ratio_mean[sub][iw, tsep, tau, ipf], yerr=ratio_err[sub][iw, tsep, tau, ipf],
                                    fmt=TSEP_MARKER[it], color=TSEP_COLOR[it], mfc=TSEP_COLOR[it] if TSEP_FILLED[it] else "none",
                                    ms=4, capsize=2, elinewidth=1, label=f"tsep {tsep}", zorder=2)
                    ax.set_title(subset_label(sub), fontsize=8)
                    ax.grid(color="#e1e0d9", lw=0.6)
                    ax.set_axisbelow(True)
                    ax.set_ylim(*ylim[ipf, w])
                    if i % n_col == 0:
                        ax.set_ylabel("R")
                    if i + n_col >= n_sub:                          # no panel below this one
                        ax.set_xlabel("tau - tsep/2")
                handles, labels = axes[0].get_legend_handles_labels()
                axes[n_sub].axis("off")
                axes[n_sub].text(0, 1, legend_text, transform=axes[n_sub].transAxes, va="top", fontsize=8)
                axes[n_sub].legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.85), frameon=False)
                for ax in axes[n_sub + 1:]:
                    ax.remove()
                kin_text = "forward q = 0, p = (0,0,pz)" if frame == "forward" else \
                           f"{frame} q = ({q[0]},{q[1]},{q[2]}), p_f = ({pf[0]},{pf[1]},{pf[2]}), p_i = ({pi[0]},{pi[1]},{pi[2]})"
                fig.suptitle(f"pz = {pf[2]}   w = {w}   {kin_text}\n{hadron} {operator_label}, hyp {hyp_use}, method 1, {DIRECTION}, {n_cfg_plot} cfg: "
                             f"ratio for every GEVP basis of {n} operator{'s' if n > 1 else ''} (t0 {t0}, tref {t_ref})", fontsize=11)
                fig.tight_layout()                                  # makes room for the suptitle too
                fig.savefig(f"{plot_root}/{n}op/w{w}/ratio_gevp_{n}op_{hadron}_{kin_tag}_pz{pf[2]}_w{w}.png", dpi=DPI)
                plt.close(fig)
            print(f"saved {plot_root}/{n}op/w{w}: pz " + ", ".join(str(p[2]) for p in pf_list), flush=True)

saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/compare_gevp_ratio_plots/sym_avg_q2_0_0-q-2_0_0-q0_2_0-q0_-2_0/ratio_by_gevp_subset/1op/w0: pz 0, 1, 2, 3, 4, 5, 6
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/compare_gevp_ratio_plots/sym_avg_q2_0_0-q-2_0_0-q0_2_0-q0_-2_0/ratio_by_gevp_subset/1op/w2: pz 0, 1, 2, 3, 4, 5, 6
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/compare_gevp_ratio_plots/sym_avg_q2_0_0-q-2_0_0-q0_2_0-q0_-2_0/ratio_by_gevp_subset/1op/w4: pz 0, 1, 2, 3, 4, 5, 6
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/compare_gevp_ratio_plots/sym_avg_q2_0_0-q-2_0_0-q0_2_0-q0_-2_0/ratio_by_gevp_subset/1op/w6: pz 0, 1, 2, 3, 4, 5, 6
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/compare_gevp_ratio_plots/sym_avg_q2_0_0-q-2_0_0-q0_2_0-q0_-2_0/ratio_by_gevp_subset/1op/w8: pz 

## 5. Choice per pz: GEVP basis and tsep range

From the plots of section 4, choose for every pz the GEVP basis (one of the subsets of sections 1-3, as its (smear, gamma) operators) and the (first, last) tsep of the fit. `CHOICE_TAG` goes into the names of the outputs of section 6, so different choices do not overwrite each other. The choice is checked and saved to `out_dir/fit_choice_{hadron}_{kin_tag}_{CHOICE_TAG}.json`, which section 6 reads.

In [52]:
#======== 5. choice per pz: the GEVP basis and the tsep range of the fit ========
CHOICE_TAG = "choice1"                                      # in the names of the outputs of section 6
CHOICE = {                                                  # pz = p_f[2]: ([(smear, gamma) of the basis], (first, last) tsep of the fit);  smear 0 = B, 1 = U;  gamma 0 = G5, 1 = G5G4, 2 = G3G5
    0: ([(0, 0),(0,1),(0,2)], (5, 18)),
    1: ([(0, 1)],(4, 14)),
    2: ([(0, 1)], (4, 14)),
    3: ([(0, 1)], (4, 11)),
    4: ([(0, 1)], (4, 10)),
    5: ([(0, 1)], (4, 10)),
    6: ([(0, 1)], (4, 10)),
}
choice_path = f"{out_dir}/fit_choice_{hadron}_{kin_tag}_{CHOICE_TAG}.json"

with h5py.File(ratio_path, "r") as f:
    pf_list = f["pf_list"][:]
choice = {}
for pf in pf_list:
    pz = int(pf[2])
    assert pz in CHOICE, f"no choice for pz = {pz}"
    basis, (first, last) = CHOICE[pz]
    assert all(op in op_all for op in basis), f"pz {pz}: {basis} has an operator that is not in op_all"
    sub = tuple(sorted(op_all.index(op) for op in basis))
    assert sub in subset_list, f"pz {pz}: the basis {subset_label(sub)} is not among the subsets of n_op_list = {n_op_list}"
    assert 0 < first < last < tsep_use, f"pz {pz}: tsep range ({first}, {last}) must lie in 1 .. tsep_use - 1 = {tsep_use - 1}"
    choice[pz] = {"subset": subset_name(sub), "label": subset_label(sub), "operators": list(sub), "basis": [list(op) for op in basis], "tsep": [first, last]}
    print(f"pz {pz}: basis {subset_label(sub):<26} tsep {first}-{last}")
with open(choice_path, "w") as f:
    json.dump({"choice": choice, "ratio_file": ratio_path, "gevp_file": gevp_path, "kin_tag": kin_tag, "frame": frame, "q": q, "hadron": hadron,
               "operator": operator, "hyp": hyp_use, "direction": DIRECTION, "choice_tag": CHOICE_TAG}, f, indent=1)
print("saved", choice_path)

pz 0: basis {B G5, B G5G4, B G3G5}     tsep 5-18
pz 1: basis {B G5G4}                   tsep 4-14
pz 2: basis {B G5G4}                   tsep 4-14
pz 3: basis {B G5G4}                   tsep 4-11
pz 4: basis {B G5G4}                   tsep 4-10
pz 5: basis {B G5G4}                   tsep 4-10
pz 6: basis {B G5G4}                   tsep 4-10
saved /Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis/final_production_delta/final_production/forward_q0_0_0/gevp_subsets/fit_choice_pion_forward_q0_0_0_choice1.json


## 6. Fits, pz by pz, with the basis and tsep range of section 5

**6a. 2pt fit** of the chosen basis of every pz: its reconstructed C2pf at $p_f$ and C2pi at $p_i$ (direction `DIRECTION`, mean over t_src; at q = 0 one fit), with the model, priors, jackknife and output layout of section 4a of the offforward notebook (3 states, $t$ = `TMIN`..`TMAX`, $E_0$ prior from the dispersion with the effective mass at `TREF`). The rest mass comes from the smallest momentum, with the basis chosen for its pz. Saved in groups `{tag}/p{px}_{py}_{pz}`, so `E_jk` is read as in `ITD.py`; each group records its basis.

**6b. Ratio fit** of every pz with its basis and tsep range: the model and settings of `fit_ratio_final_offforward.py` ($M_{00} + A_i e^{-\Delta E_i\tau} + A_f e^{-\Delta E_f(t_{\rm sep}-\tau)} + A_{fi} e^{-\Delta E_i\tau-\Delta E_f(t_{\rm sep}-\tau)}$, at q = 0 $A_f = A_i$ and $\Delta E_f = \Delta E_i$), $\tau$ = `TAU_SKIP`..tsep−`TAU_SKIP`, jackknife covariance, a central fit and one fit per jackknife sample with the gap prior of that sample from 6a (initial gap at $p_i$, final gap at $p_f$, width `DE_WIDTH_FACTOR` × that sample's lsqfit error), or `DE_PRIOR` for every sample when it is set. Each w is fitted alone, which is the script's fit with `CROSS_W_FACTOR = 0`. 6b uses `FIT_PATH` and `tag` of 6a. Saved per pz in the script's layout (`bare_matrix_element_jk` (jk, w), `w_list`, ...), with a plot of the fit per pz.

In [42]:
#======== 6a. 2pt fit of the chosen basis of every pz (model of the offforward notebook's 4a) ========
import gvar as gv
import lsqfit

Gs, Gt = 32, 96
TMIN, TMAX = 5, 15
TSEP_LIST = [4, 5, 6, 7]               # (c1 e^-E1 tsep) / (c0 e^-E0 tsep) is printed
N_STATES = 4
SVDCUT = 1e-7
TREF = 10                              # E0 prior is read off the effective mass at this t
MAXIT = 20000
DE1_PRIOR = gv.gvar("0.50(0.40)")
DE2_PRIOR = gv.gvar("0.50(0.40)")
C_CENTER = 0.0
C_WIDTH = 1e4
PROGRESS_EVERY = 100
tag = f"nstate{N_STATES}_t{TMIN}-{TMAX}_svd{SVDCUT:.0e}_tref{TREF}"
with open(f"{out_dir}/fit_choice_{hadron}_{kin_tag}_{CHOICE_TAG}.json") as f:
    choice = {int(pz): c for pz, c in json.load(f)["choice"].items()}
FIT_PATH = f"{out_dir}/twopt_fit_results_{hadron}_{kin_tag}_{CHOICE_TAG}_{DIRECTION}_{run_tag}.h5"


def twopt_model(t, p):
    e = p["E0"]
    corr = p["c"][0] * (gv.exp(-e * t) + gv.exp(-e * (Gt - t)))
    for n in range(1, len(p["c"])):
        e = e + p["dE"][n - 1]
        corr = corr + p["c"][n] * (gv.exp(-e * t) + gv.exp(-e * (Gt - t)))
    return corr


def make_prior_2pt(n_states, e0_center, e0_width):
    gaps = [DE1_PRIOR, DE2_PRIOR][:n_states - 1]
    prior = {"E0": gv.gvar(e0_center, e0_width), "c": gv.gvar([C_CENTER] * n_states, [C_WIDTH] * n_states)}
    if n_states > 1:
        prior["log(dE)"] = gv.log(gv.gvar([gv.mean(g) for g in gaps], [gv.sdev(g) for g in gaps]))
    return prior


def cov_of_mean(samples):
    return np.cov(samples, rowvar=False, ddof=1) / samples.shape[0]


def jk_stats(values):
    n = values.shape[0]
    mean = values.mean(axis=0)
    return mean, np.sqrt((n - 1.0) / n * np.sum((values - mean) ** 2, axis=0))


def effmass_at_tref(raw):
    """acosh[(C(t+1) + C(t-1)) / (2 C(t))] at t = TREF on the delete-one replicates of raw [n_cfg, t]: (jk_mean, jk_err)"""
    n = raw.shape[0]
    reps = (raw.sum(axis=0) - raw) / (n - 1.0)
    em = np.arccosh((reps[:, TREF + 1] + reps[:, TREF - 1]) / (2.0 * reps[:, TREF]))
    return jk_stats(em)


#---- the reconstructed C2pf (at p_f) and C2pi (at p_i) of the chosen basis of every pz, this direction, mean over t_src: (cfg, tsep); every momentum once
fit_list = []                                           # (momentum, pz of the choice, 2pt)
with h5py.File(gevp_path, "r") as f:
    for ip, (pf, pi) in enumerate(zip(f["pf_list"][:], f["pi_list"][:])):
        pz = int(pf[2])
        g = f[choice[pz]["subset"]]
        for role, p in [("C2pf", pf), ("C2pi", pi)]:
            per_dir = {d: g[f"pt2_{role}_{d}"][:, ip, :].real for d in ("forward", "backward")}
            twopt = per_dir[DIRECTION] if DIRECTION != "fb" else 0.5 * (per_dir["forward"] + per_dir["backward"])
            p = tuple(int(v) for v in p)
            same = [item for item in fit_list if item[0] == p]
            if same:                                    # at q = 0 p_i = p_f; any other repeat must have the same basis
                assert choice[same[0][1]]["subset"] == choice[pz]["subset"], f"p = {p} is needed with two bases: pz {same[0][1]} and pz {pz}"
                continue
            fit_list.append((p, pz, twopt))
t = np.arange(TMIN, TMAX + 1)
n_jk = fit_list[0][2].shape[0]

#rest mass: effective mass at t = TREF of the smallest momentum (with the basis of its pz), through the continuum dispersion
p_min, pz_min, C2_min = min(fit_list, key=lambda item: sum(c ** 2 for c in item[0]))
E_min, _ = effmass_at_tref(C2_min)
m_rest = np.sqrt(E_min ** 2 - (2.0 * np.pi / Gs) ** 2 * sum(c ** 2 for c in p_min))
print(f"rest mass from effective mass at t = {TREF} and p = {p_min} ({choice[pz_min]['label']}): {m_rest:.6f}", flush=True)

for pf, pz, C2_pf in fit_list:
    samples = C2_pf[:, TMIN:TMAX + 1]
    mean = samples.mean(axis=0)
    cov = cov_of_mean(samples)
    jk_samples = (samples.sum(axis=0) - samples) / (n_jk - 1)

    e0_center = np.sqrt(m_rest ** 2 + (2.0 * np.pi / Gs) ** 2 * sum(c ** 2 for c in pf))
    _, e0_width = effmass_at_tref(C2_pf)
    prior = make_prior_2pt(N_STATES, e0_center, e0_width)

    central_fit = lsqfit.nonlinear_fit(data=(t, gv.gvar(mean, cov)), fcn=twopt_model, prior=prior, svdcut=SVDCUT, maxit=MAXIT)
    dE_full = np.concatenate([np.atleast_1d(central_fit.p["E0"]), central_fit.p["dE"]])
    E = np.cumsum(dE_full)
    E_central, E_central_sdev = gv.mean(E), gv.sdev(E)
    C_central, C_central_sdev = gv.mean(central_fit.p["c"]), gv.sdev(central_fit.p["c"])
    chi2dof_central, Q_central = central_fit.chi2 / central_fit.dof, central_fit.Q

    print(f"\n====== CENTRAL FIT RESULT   p = {pf}   basis {choice[pz]['label']} (pz {pz}) ======")
    print(f"  central: n_states = {N_STATES}   t = {TMIN}..{TMAX}")
    print(f"  central: E0 prior = {gv.gvar(e0_center, e0_width)}   (dispersion center, effmass width at t = {TREF})")
    for n in range(N_STATES):
        print(f"  central: E{n} = {E[n]:<14} c{n} = {central_fit.p['c'][n]}")
    print(f"  central: chi2/dof = {chi2dof_central:.2f}   Q = {Q_central:.3f}   svdn = {getattr(central_fit, 'svdn', 0)}")
    ratio = central_fit.p["c"][1] / central_fit.p["c"][0] * gv.exp(-central_fit.p["dE"][0] * np.array(TSEP_LIST))
    for tsep, r in zip(TSEP_LIST, ratio):
        print(f"  central: C1 e^(-E1 tsep) / C0 e^(-E0 tsep) at tsep = {tsep}: {r}")

    #per jackknife sample: central values and each replicate's own fit error (slot 0 of dE_jk is E0, slots 1+ are gaps)
    E_jk, dE_jk, C_jk = np.zeros((n_jk, N_STATES)), np.zeros((n_jk, N_STATES)), np.zeros((n_jk, N_STATES))
    E_sdev_jk, dE_sdev_jk, C_sdev_jk = np.zeros((n_jk, N_STATES)), np.zeros((n_jk, N_STATES)), np.zeros((n_jk, N_STATES))
    chi2dof_jk, Q_jk = np.zeros(n_jk), np.zeros(n_jk)
    p0 = {k: central_fit.pmean[k] for k in prior}
    for i in range(n_jk):
        fit_jk = lsqfit.nonlinear_fit(data=(t, gv.gvar(jk_samples[i], cov)), fcn=twopt_model, prior=prior, svdcut=SVDCUT, p0=p0, maxit=MAXIT)
        dE_i = np.concatenate([np.atleast_1d(fit_jk.p["E0"]), fit_jk.p["dE"]])
        E_i = np.cumsum(dE_i)
        dE_jk[i], E_jk[i], C_jk[i] = gv.mean(dE_i), gv.mean(E_i), gv.mean(fit_jk.p["c"])
        E_sdev_jk[i], dE_sdev_jk[i], C_sdev_jk[i] = gv.sdev(E_i), gv.sdev(dE_i), gv.sdev(fit_jk.p["c"])
        chi2dof_jk[i], Q_jk[i] = fit_jk.chi2 / fit_jk.dof, fit_jk.Q
        if PROGRESS_EVERY and (i + 1) % PROGRESS_EVERY == 0:
            print(f"  jackknife {i + 1}/{n_jk}", flush=True)

    dE_jk_mean, dE_jk_err = jk_stats(dE_jk)
    E_jk_mean, E_jk_err = jk_stats(E_jk)
    print(f"  jackknife: E0 = {dE_jk_mean[0]:.6f}({dE_jk_err[0]:.6f})   dE1 = {dE_jk_mean[1]:.4f}({dE_jk_err[1]:.4f})"
          f"   <chi2/dof> = {chi2dof_jk.mean():.2f}   <Q> = {Q_jk.mean():.3f}")

    with h5py.File(FIT_PATH, "a") as f:
        g = f.require_group(tag)
        g.attrs["tmin"], g.attrs["tmax"], g.attrs["n_states"], g.attrs["svdcut"], g.attrs["n_jk"], g.attrs["tref"] = TMIN, TMAX, N_STATES, SVDCUT, n_jk, TREF
        g.attrs["E0_prior_scheme"] = f"center sqrt(m_rest^2 + p^2), m_rest = effmass(t={TREF}) at the smallest momentum; width = this momentum's effmass jk error"
        g.attrs["pt2_source"] = gevp_path
        g.attrs["direction"] = DIRECTION
        g.attrs["choice"] = json.dumps(choice)
        mom_name = f"p{pf[0]}_{pf[1]}_{pf[2]}"
        if mom_name in g:
            del g[mom_name]
        gm = g.create_group(mom_name)
        for name, data in [("momentum", np.array(pf, dtype=np.int64)), ("E_central", E_central), ("E_central_sdev", E_central_sdev),
                           ("C_central", C_central), ("C_central_sdev", C_central_sdev), ("chi2dof_central", chi2dof_central),
                           ("Q_central", Q_central), ("E_jk", E_jk), ("dE_jk", dE_jk), ("C_jk", C_jk), ("E_sdev_jk", E_sdev_jk),
                           ("dE_sdev_jk", dE_sdev_jk), ("C_sdev_jk", C_sdev_jk), ("chi2dof_jk", chi2dof_jk), ("Q_jk", Q_jk),
                           ("E_jk_mean", E_jk_mean), ("E_jk_err", E_jk_err), ("dE_jk_mean", dE_jk_mean), ("dE_jk_err", dE_jk_err)]:
            gm.create_dataset(name, data=data)
        gm.attrs["E0_prior_center"] = e0_center
        gm.attrs["E0_prior_width"] = e0_width
        gm.attrs["basis"] = choice[pz]["label"]
        gm.attrs["basis_pz"] = pz
        gm.attrs["dim_E_jk"] = "jk,state"
        gm.attrs["dim_dE_jk"] = "jk,state (slot 0 is E0, slots 1+ are gaps)"
        gm.attrs["dim_dE_sdev_jk"] = "jk,state -- lsqfit's uncertainty on each replicate's own fit, NOT the jackknife scatter of dE_jk across replicates"
    print(f"  written: {FIT_PATH}:{tag}/{mom_name}", flush=True)

NameError: name 'CHOICE_TAG' is not defined

In [ ]:
#======== 6b. ratio fit of every pz with its basis and tsep range (model of fit_ratio_final_offforward.py, each w alone) ========
TAU_SKIP = 2                           # tau = TAU_SKIP .. tsep - TAU_SKIP
DE_WIDTH_FACTOR = 1.0                  # gap prior width = DE_WIDTH_FACTOR x the 2pt fit's lsqfit error of that jackknife sample
DE_PRIOR = None                        # None: gap prior from the 2pt fit of 6a per jackknife sample; or e.g. gv.gvar("0.5(5)") for every sample and pz
A_PRIOR_WIDTH = 1e4
M00_PRIOR_WIDTH = 1e4
SVDCUT = 1e-07
MAXIT = 20000
w_fit_plot_list = [0, 1, 2, 3, 4, 5, 6, 8]
with open(f"{out_dir}/fit_choice_{hadron}_{kin_tag}_{CHOICE_TAG}.json") as f:
    choice = {int(pz): c for pz, c in json.load(f)["choice"].items()}
forward = all(c == 0 for c in q)
fit_tag = (f"svd{SVDCUT:.0e}_dEf{DE_WIDTH_FACTOR:g}_xw0_tauskip{TAU_SKIP}"
           + ("" if DE_PRIOR is None else f"_dEprior{DE_PRIOR.mean:g}w{DE_PRIOR.sdev:g}")).replace(".", "p")
OUTPUT_DIR = f"{out_dir}/bare_matrix_element_{kin_tag}_{hadron}_{CHOICE_TAG}_method1_{DIRECTION}_2pt_{tag}_perjk"
PLOT_DIR = f"{out_dir}/ratio_fit_plots_{kin_tag}_{hadron}_{CHOICE_TAG}_method1_{DIRECTION}_{tag}"
for folder in [OUTPUT_DIR, PLOT_DIR]:
    os.makedirs(folder, exist_ok=True)


def ratio_model(x, p):
    """x = (tsep, tau) of every data point, p = the parameters of one w"""
    tau, tsep_tau = x["tau"], x["tsep"] - x["tau"]
    dEf = p["dEf"] if "log(dEf)" in p else p["dEi"]
    Af = p["Af"] if "Af" in p else p["Ai"]
    return p["M00"] + p["Ai"] * gv.exp(-p["dEi"] * tau) + Af * gv.exp(-dEf * tsep_tau) + p["Afi"] * gv.exp(-p["dEi"] * tau - dEf * tsep_tau)


def make_prior(gap_i, width_i, gap_f, width_f):
    prior = gv.BufferDict()
    prior["M00"] = gv.gvar(0.0, M00_PRIOR_WIDTH)
    prior["Ai"] = gv.gvar(0.0, A_PRIOR_WIDTH)
    prior["Afi"] = gv.gvar(0.0, A_PRIOR_WIDTH)
    prior["log(dEi)"] = gv.log(DE_PRIOR if DE_PRIOR is not None else gv.gvar(gap_i, width_i * DE_WIDTH_FACTOR))
    if not forward:
        prior["Af"] = gv.gvar(0.0, A_PRIOR_WIDTH)
        prior["log(dEf)"] = gv.log(DE_PRIOR if DE_PRIOR is not None else gv.gvar(gap_f, width_f * DE_WIDTH_FACTOR))
    return prior


with h5py.File(ratio_path, "r") as f:
    pf_list, pi_list = f["pf_list"][:], f["pi_list"][:]
    w_list = f["w_list"][:].tolist()
    n_cfg_fit = len(f["cfg_list"][:])
names = ["M00", "Ai", "Afi", "dEi"] + ([] if forward else ["Af", "dEf"])
for ipf, (pf, pi) in enumerate(zip(pf_list, pi_list)):
    pz = int(pf[2])
    first, last = choice[pz]["tsep"]
    with h5py.File(ratio_path, "r") as f:
        ratio_jk = f[choice[pz]["subset"]]["ratio_jk"][:, :, :, :, ipf]    # (jk, w, tsep, tau)
    n_jk = ratio_jk.shape[0]
    points = [(tsep, tau) for tsep in range(first, last + 1) for tau in range(TAU_SKIP, tsep - TAU_SKIP + 1)]
    x = {"tsep": np.array([float(T) for T, _ in points]), "tau": np.array([float(t) for _, t in points])}

    #gap priors from the 2pt fit at p_f and at p_i = p_f - q, per jackknife sample
    with h5py.File(FIT_PATH, "r") as f:
        gf, gi = f[tag][f"p{pf[0]}_{pf[1]}_{pf[2]}"], f[tag][f"p{pi[0]}_{pi[1]}_{pi[2]}"]
        assert gf.attrs["basis"] == choice[pz]["label"] and gi.attrs["basis"] == choice[pz]["label"], f"pz {pz}: the 2pt fit in {FIT_PATH} is of another basis, run 6a again"
        gap_f_jk, gap_f_err_jk = gf["dE_jk"][:, 1], gf["dE_sdev_jk"][:, 1]
        gap_i_jk, gap_i_err_jk = gi["dE_jk"][:, 1], gi["dE_sdev_jk"][:, 1]
    assert len(gap_f_jk) == n_jk, f"2pt has {len(gap_f_jk)} samples, the ratio has {n_jk}"
    gap_f_mean, gap_f_err = jk(gap_f_jk)
    gap_i_mean, gap_i_err = jk(gap_i_jk)
    print(f"pz {pz}, basis {choice[pz]['label']}, tsep {first}-{last}: {len(points)} points per w, {n_jk} samples, 2pt gap "
          f"pi {tuple(int(v) for v in pi)} {gap_i_mean:.3f} +- {gap_i_err:.3f}, pf {tuple(int(v) for v in pf)} {gap_f_mean:.3f} +- {gap_f_err:.3f}"
          + ("" if DE_PRIOR is None else f";  gap prior {DE_PRIOR} instead"), flush=True)

    params = {n: np.zeros((n_jk, len(w_list))) for n in names}
    chi2dof, chi2dof_central, Q = np.zeros((n_jk, len(w_list))), np.zeros(len(w_list)), np.zeros((n_jk, len(w_list)))
    samples_w, restarted = {}, []
    for iw, w in enumerate(w_list):
        samples = np.stack([ratio_jk[:, iw, tsep, tau] for tsep, tau in points], axis=1)        # (jk, point)
        samples_w[w] = samples
        mean = samples.mean(axis=0)
        cov = np.cov(samples, rowvar=False, ddof=1) * (n_jk - 1.0) ** 2 / n_jk                   # jackknife covariance
        central_fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(mean, cov)), fcn=ratio_model, svdcut=SVDCUT, maxit=MAXIT,
                                           prior=make_prior(gap_i_mean, gap_i_err, gap_f_mean, gap_f_err))
        chi2dof_central[iw] = central_fit.chi2 / central_fit.dof
        p0 = {k: central_fit.pmean[k] for k in central_fit.prior}
        for i in range(n_jk):
            prior_jk = make_prior(gap_i_jk[i], gap_i_err_jk[i], gap_f_jk[i], gap_f_err_jk[i])
            try:
                fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(samples[i], cov)), fcn=ratio_model, svdcut=SVDCUT, maxit=MAXIT, prior=prior_jk, p0=p0)
            except np.linalg.LinAlgError:
                #rare: scipy's SVD fails on the way from the central-fit start; the same fit restarted from the prior means (p0 left out)
                restarted.append((w, i))
                fit = lsqfit.nonlinear_fit(data=(x, gv.gvar(samples[i], cov)), fcn=ratio_model, svdcut=SVDCUT, maxit=MAXIT, prior=prior_jk)
            for n in names:
                params[n][i, iw] = gv.mean(fit.p[n])
            chi2dof[i, iw], Q[i, iw] = fit.chi2 / fit.dof, fit.Q
    if restarted:
        print(f"  (w, jackknife sample) restarted from the prior means: {restarted}")

    M00_mean, M00_err = jk(params["M00"])
    print(f"  M00 at w = {', '.join(str(w) for w in w_list)}: " + "  ".join(f"{gv.gvar(m, e)}" for m, e in zip(M00_mean, M00_err))
          + f"\n  <chi2/dof> per w: " + " ".join(f"{c:.2f}" for c in chi2dof.mean(axis=0)), flush=True)
    out = f"{OUTPUT_DIR}/bareM_{operator}_{frame}_hyp{hyp_use}_pf{pf[0]}_{pf[1]}_{pf[2]}_q{q[0]}_{q[1]}_{q[2]}_tsep{first}-{last}_{fit_tag}.h5"
    with h5py.File(out, "w") as f:
        f.create_dataset("bare_matrix_element_jk", data=params["M00"])
        for n in names:
            f.create_dataset(f"{n}_jk", data=params[n])
        f.create_dataset("M00_jk_mean", data=M00_mean)
        f.create_dataset("M00_jk_err", data=M00_err)
        f.create_dataset("chi2dof_jk", data=chi2dof)
        f.create_dataset("w_list", data=np.array(w_list, dtype=np.int64))
        f.create_dataset("gap_i_prior_center_jk", data=gap_i_jk)
        f.create_dataset("gap_f_prior_center_jk", data=gap_f_jk)
        f.attrs["dim_bare_matrix_element_jk"] = "jk,w_index"
        f.attrs["operator"] = operator
        f.attrs["hadron"] = hadron
        f.attrs["basis"] = choice[pz]["label"]
        f.attrs["basis_operators"] = json.dumps(choice[pz]["basis"])
        f.attrs["gevp"] = f"t0 {t0}, t_ref {t_ref}"
        f.attrs["twopt_tag"] = tag
        f.attrs["twopt_fit"] = FIT_PATH
        f.attrs["frame"] = frame
        f.attrs["hyp"] = hyp_use
        f.attrs["pf"] = np.array(pf, dtype=np.int64)
        f.attrs["pi"] = np.array(pi, dtype=np.int64)
        f.attrs["q"] = np.array(q, dtype=np.int64)
        f.attrs["gap_prior"] = (f"2pt E1-E0 per sample, width {DE_WIDTH_FACTOR} x that sample's lsqfit error" if DE_PRIOR is None else f"{DE_PRIOR} for every sample")
        f.attrs["tsep_fit_range"] = np.array([first, last], dtype=np.int64)
        f.attrs["tau_skip"] = TAU_SKIP
        f.attrs["svdcut"] = SVDCUT
        f.attrs["cross_w_factor"] = 0.0
        f.attrs["fit_per_w"] = "each w fitted alone (the same as fit_ratio_final_offforward.py with CROSS_W_FACTOR = 0)"
        f.attrs["direction"] = DIRECTION
        f.attrs["vacuum_method"] = 1
        f.attrs["ratio_source"] = ratio_path
        f.attrs["choice_tag"] = CHOICE_TAG
        f.attrs["jackknife_restarted"] = json.dumps(restarted)        # (w, sample) refitted from the prior means
        f.attrs["chi2dof"] = float(chi2dof.mean())
        f.attrs["Q"] = float(Q.mean())
    print(f"  saved {out}", flush=True)

    #---- plot: one panel per w, the fitted points and the fit band of every tsep, M00 in gray
    ws = [w for w in w_fit_plot_list if w in w_list]
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, len(ws), figsize=(3.75 * len(ws), 3.9), squeeze=False, sharey=True)
        for icol, w in enumerate(ws):
            ax, iw = axes[0][icol], w_list.index(w)
            data_mean, data_err = jk(samples_w[w])
            ax.axhspan(M00_mean[iw] - M00_err[iw], M00_mean[iw] + M00_err[iw], color="#e7e7e5", lw=0, zorder=0)
            ax.axhline(0, color="#52514e", lw=0.8, zorder=1)
            for tsep in range(first, last + 1):
                it = tsep - tsep_style_first
                sel = x["tsep"] == tsep
                ax.errorbar(x["tau"][sel] - 0.5 * tsep, data_mean[sel], yerr=data_err[sel], fmt=TSEP_MARKER[it], color=TSEP_COLOR[it],
                            mfc=TSEP_COLOR[it] if TSEP_FILLED[it] else "none", ms=4, capsize=2, elinewidth=1, label=f"tsep {tsep}", zorder=2)
                #the band: the fitted curve on every jackknife sample
                tau = np.linspace(TAU_SKIP, tsep - TAU_SKIP, 60)[None, :]
                de, ai, afi = params["dEi"][:, iw][:, None], params["Ai"][:, iw][:, None], params["Afi"][:, iw][:, None]
                df = params["dEf"][:, iw][:, None] if "dEf" in params else de
                af = params["Af"][:, iw][:, None] if "Af" in params else ai
                curve = params["M00"][:, iw][:, None] + ai * np.exp(-de * tau) + af * np.exp(-df * (tsep - tau)) + afi * np.exp(-de * tau - df * (tsep - tau))
                b_mean, b_err = jk(curve)
                ax.fill_between(tau[0] - 0.5 * tsep, b_mean - b_err, b_mean + b_err, color=TSEP_COLOR[it], alpha=0.25, lw=0, zorder=1)
            ax.set_title(f"w = {w}   M00 = {gv.gvar(M00_mean[iw], M00_err[iw])}   <chi2/dof> {chi2dof[:, iw].mean():.2f}", fontsize=9)
            ax.set_xlabel("tau - tsep/2")
            ax.grid(color="#e1e0d9", lw=0.6)
            ax.set_axisbelow(True)
        axes[0][0].set_ylabel("R")
        axes[0][-1].legend(fontsize=7)
        fig.suptitle(f"{hadron} {operator_label}   {kin_tag}   p_f = ({pf[0]},{pf[1]},{pf[2]})   basis {choice[pz]['label']} (t0 {t0}, tref {t_ref})   "
                     f"tsep {first}-{last}, tau skip {TAU_SKIP}   hyp {hyp_use}   method 1   {DIRECTION}   {n_cfg_fit} cfg", fontsize=11)
        fig.tight_layout()
        fig.savefig(f"{PLOT_DIR}/fit_{operator}_{frame}_hyp{hyp_use}_pf{pf[0]}_{pf[1]}_{pf[2]}_q{q[0]}_{q[1]}_{q[2]}_tsep{first}-{last}_{fit_tag}_{DIRECTION}.png", dpi=150)
        plt.close(fig)
print("bare matrix elements in", OUTPUT_DIR)
print("plots in", PLOT_DIR)